In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
prices = pd.read_csv("sell_prices.csv")
calendar = pd.read_csv("calendar.csv")
sales = pd.read_csv("sales_train_evaluation.csv")

In [ ]:
#Make assumption that food items will fluctuate the most
foods_items = sales[sales["cat_id"] == "FOODS"]["item_id"].unique()
foods_prices = prices[prices["item_id"].isin(foods_items)]

In [ ]:
#  Compute price volatility per item (across all stores)
#    Higher coefficient of variation (CV) = more price movement = promo candidate
price_stats = (
    foods_prices.groupby("item_id")["sell_price"]
    .agg(mean_price="mean", std_price="std", n_price_points="count")
    .reset_index()
)
price_stats["cv"] = price_stats["std_price"] / price_stats["mean_price"]

# Require enough price history to be meaningful
price_stats = price_stats[price_stats["n_price_points"] > 50]

# Top 20 most price volatile food items
top_volatile = price_stats.sort_values("cv", ascending=False).head(20)
print("Top 20 price-volatile FOODS items:")
print(top_volatile[["item_id", "mean_price", "cv", "n_price_points"]])


In [ ]:
# For a shortlist of candidates, check if sales actually spike
#    when price drops (not just that price is volatile)

def demand_price_correlation(item_id, store_id="CA_1"):
    """
    Returns the correlation between weekly price and weekly sales
    for one item at one store. Strong negative correlation = 
    demand rises when price falls = good promo candidate.
    """
    # Sales row for this item/store
    sales_row = sales[
        (sales["item_id"] == item_id) & (sales["store_id"] == store_id)
    ]
    if sales_row.empty:
        return None

    day_cols = [c for c in sales.columns if c.startswith("d_")]
    daily_sales = sales_row[day_cols].T
    daily_sales.columns = ["units_sold"]
    daily_sales["d"] = daily_sales.index

    # Map d_ columns to actual weeks via calendar
    daily_sales = daily_sales.merge(calendar[["d", "wm_yr_wk"]], on="d")
    weekly_sales = daily_sales.groupby("wm_yr_wk")["units_sold"].sum().reset_index()

    item_prices = prices[
        (prices["item_id"] == item_id) & (prices["store_id"] == store_id)
    ][["wm_yr_wk", "sell_price"]]

    merged = weekly_sales.merge(item_prices, on="wm_yr_wk")
    if len(merged) < 10:
        return None

    corr = merged["units_sold"].corr(merged["sell_price"])
    return corr

print("\nChecking demand-price correlation for top candidates (store CA_1):")
results = []
for item_id in top_volatile["item_id"].head(10):
    corr = demand_price_correlation(item_id)
    if corr is not None:
        results.append((item_id, corr))

results_df = pd.DataFrame(results, columns=["item_id", "price_sales_corr"])
results_df = results_df.sort_values("price_sales_corr")  # most negative first
print(results_df)

print("""
How to read this:
- price_sales_corr close to -1: strong promo pattern (sales rise sharply as price drops)
- price_sales_corr near 0: little relationship (steady seller, not promo-driven)
- Pick 1-2 items from the top of results_df (most negative correlation) as your
  forecasting subjects.
""")


In [ ]:

# Check whether the promo pattern for our chosen item holds
#    across ALL stores, not just CA_1

CHOSEN_ITEM = "FOODS_3_305"
all_stores = sales["store_id"].unique()
 
store_results = []
for store_id in all_stores:
    corr = demand_price_correlation(CHOSEN_ITEM, store_id=store_id)
    if corr is not None:
        store_results.append((store_id, corr))
 
store_results_df = pd.DataFrame(store_results, columns=["store_id", "price_sales_corr"])
store_results_df = store_results_df.sort_values("price_sales_corr")
 
print(f"\nPrice-sales correlation for {CHOSEN_ITEM} across all stores:")
print(store_results_df)
print(f"\nMean correlation across stores: {store_results_df['price_sales_corr'].mean():.3f}")
print(f"Std dev across stores: {store_results_df['price_sales_corr'].std():.3f}")
 
print("""
How to read this:
- If most/all stores show a similar negative correlation: the promo pattern is
  real and consistent, aggregating nationwide is reasonable.
- If correlations vary widely (some strongly negative, some near zero or positive):
  the pattern is store/region-specific, worth noting in your writeup, and you may
  want to pick a single representative store (or a cluster of similar stores)
  rather than aggregating everything together.
""")
 
